In [2]:
import pandas as pd
from sklearn.impute import KNNImputer

# 1. Load dataset gabungan (3 gunung)
df = pd.read_csv('vona reports selain merapi_extracted.csv')

# 2. Feature engineering kategori -> numerik
alert_map = {'Green': 0, 'Yellow': 1, 'Orange': 2, 'Red': 3}
df['alert_lvl_num'] = df['alert_lvl'].map(alert_map)

# One-hot encoding nama gunung (contoh: volcano_Bromo, volcano_Raung, ...)
volcano_dummies = pd.get_dummies(df['volcano_filter'], prefix='volcano')
df = pd.concat([df, volcano_dummies], axis=1)

# 3. Hanya kolom ini yang akan diimpute
cols_to_impute = ['amplitudo', 'max_time']

# Fitur bantu untuk hitung kemiripan tetangga KNN (tidak diubah nilainya)
helper_feature_cols = [
    'lat',
    'long',
    'elevation',
    'tinggi_letusan_m',
    'kec_angin_km_jam',
    'arah_angin_deg',
    'alert_lvl_num',
] + volcano_dummies.columns.tolist()

# Input KNN = fitur bantu + kolom target imputasi
imputer_input_cols = helper_feature_cols + cols_to_impute
imputer_input = df[imputer_input_cols].apply(pd.to_numeric, errors='coerce')

# 4. Inisialisasi dan jalankan imputasi
imputer = KNNImputer(n_neighbors=3, keep_empty_features=True)
imputed_array = imputer.fit_transform(imputer_input)
imputed_df = pd.DataFrame(imputed_array, columns=imputer_input_cols, index=df.index)

# 5. Kembalikan hanya kolom target imputasi ke dataframe asli
for col in cols_to_impute:
    df[col] = imputed_df[col]

# Simpan hasil
output_file = 'vona_3gunung_knn_imputed.csv'
df.to_csv(output_file, index=False)

print(f'Output tersimpan: {output_file}')
print('Kolom diimputasi:', cols_to_impute)
print('Kolom lain dibiarkan sesuai data asli (tetap kosong jika kosong).')
df[['volcano_filter', 'amplitudo', 'max_time']].head()

Output tersimpan: vona_3gunung_knn_imputed.csv
Kolom diimputasi: ['amplitudo', 'max_time']
Kolom lain dibiarkan sesuai data asli (tetap kosong jika kosong).


,volcano_filter,amplitudo,max_time
0,Bromo,8.666667,124.0
1,Bromo,8.000000,124.0
2,Bromo,8.666667,124.0
3,Bromo,8.666667,124.0
4,Bromo,8.666667,124.0
